In [24]:
from pathlib import Path
import shutil

# === EDIT THIS PATH to your parent dataset folder ===
ROOT = Path(r"/Users/vanhorin/Documents/Wang/Paired dataset_10122025")

# File types to include
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

# Safety toggle: start with DRY_RUN=True to preview without copying
DRY_RUN = False  # set to False to actually copy

# Non-recursive by default. Set to True if Sickled/Non-sickled contain subfolders you want to include.
RECURSIVE = False

# Destination directories
ALL_DIR = ROOT / "All"
DST_SICKLED = ALL_DIR / "Sickled"
DST_NON = ALL_DIR / "Non-sickled"
DST_SICKLED.mkdir(parents=True, exist_ok=True)
DST_NON.mkdir(parents=True, exist_ok=True)

def want_type_dir(p: Path) -> bool:
    # Accept any top-level folder except 'All'
    return p.is_dir() and p.name.lower() != "all"

def safe_name(prefix: str, name: str) -> str:
    # Avoid collisions by prefixing the source type folder (e.g., 'D__file.png')
    return f"{prefix}__{name}"

def copy_one(src: Path, dst_dir: Path, type_prefix: str):
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / safe_name(type_prefix, src.name)

    # If even the prefixed name exists, append numeric suffix
    if dst.exists():
        stem, ext = dst.stem, dst.suffix
        k = 1
        while True:
            candidate = dst_dir / f"{stem}__{k}{ext}"
            if not candidate.exists():
                dst = candidate
                break
            k += 1

    print(f"COPY: {src} -> {dst}")
    if not DRY_RUN:
        shutil.copy2(src, dst)

def iter_images(folder: Path):
    if RECURSIVE:
        for f in folder.rglob("*"):
            if f.is_file() and f.suffix.lower() in IMAGE_EXTS:
                yield f
    else:
        for f in folder.iterdir():
            if f.is_file() and f.suffix.lower() in IMAGE_EXTS:
                yield f

print("Root:", ROOT)
print("Dry run:", DRY_RUN)
print("Recursive:", RECURSIVE)


Root: /Users/vanhorin/Documents/Wang/Paired dataset_10122025
Dry run: False
Recursive: False


In [25]:
skipped = []
type_dirs = [d for d in ROOT.iterdir() if want_type_dir(d)]

if not type_dirs:
    print("No type directories found.")
else:
    for tdir in sorted(type_dirs):
        type_name = tdir.name
        subs = {sd.name.lower(): sd for sd in tdir.iterdir() if sd.is_dir()}
        sickled_dir = next((p for n, p in subs.items() if n.startswith("sickled")), None)
        nonsic_dir  = next((p for n, p in subs.items() if n.startswith("non-sickled")), None)

        if sickled_dir is None and nonsic_dir is None:
            print(f"[WARN] {tdir} has no Sickled/Non-sickled; skipping.")
            skipped.append(tdir)
            continue

        if sickled_dir and sickled_dir.exists():
            for f in iter_images(sickled_dir):
                copy_one(f, DST_SICKLED, type_name)

        if nonsic_dir and nonsic_dir.exists():
            for f in iter_images(nonsic_dir):
                copy_one(f, DST_NON, type_name)

    print("\nDone. If everything looks right, set DRY_RUN=False and run again.")
    if skipped:
        print("Skipped folders:")
        for s in skipped:
            print("  -", s)


COPY: /Users/vanhorin/Documents/Wang/Paired dataset_10122025/A/Sickled/2_trial_2_id141.0_frame0.png -> /Users/vanhorin/Documents/Wang/Paired dataset_10122025/All/Sickled/A__2_trial_2_id141.0_frame0.png
COPY: /Users/vanhorin/Documents/Wang/Paired dataset_10122025/A/Sickled/6th_fresh_trial1_4PFS_id111.0_frame1.png -> /Users/vanhorin/Documents/Wang/Paired dataset_10122025/All/Sickled/A__6th_fresh_trial1_4PFS_id111.0_frame1.png
COPY: /Users/vanhorin/Documents/Wang/Paired dataset_10122025/A/Sickled/2193-osi-30per-4_id49.0_frame1.png -> /Users/vanhorin/Documents/Wang/Paired dataset_10122025/All/Sickled/A__2193-osi-30per-4_id49.0_frame1.png
COPY: /Users/vanhorin/Documents/Wang/Paired dataset_10122025/A/Sickled/fresh-1_id133.0_frame0.png -> /Users/vanhorin/Documents/Wang/Paired dataset_10122025/All/Sickled/A__fresh-1_id133.0_frame0.png
COPY: /Users/vanhorin/Documents/Wang/Paired dataset_10122025/A/Sickled/2nd_microfluidic-3_id53.0_frame1.png -> /Users/vanhorin/Documents/Wang/Paired dataset_101

In [26]:
from pathlib import Path

# === EDIT THIS to your dataset root ===
ROOT = Path(r"/Users/vanhorin/Documents/Wang/Paired dataset_10122025")

# Count only these extensions (case-insensitive)
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

# If your Sickled/Non-sickled contain nested subfolders, set RECURSIVE=True
RECURSIVE = False

def is_type_dir(p: Path) -> bool:
    return p.is_dir() and p.name.lower() != "all"

def find_class_dirs(type_dir: Path):
    subs = [sd for sd in type_dir.iterdir() if sd.is_dir()]
    sickled   = next((sd for sd in subs if sd.name.lower().startswith("sickled")), None)
    nonsickle = next((sd for sd in subs if sd.name.lower().startswith("non-sickled")), None)
    return sickled, nonsickle

def count_images(folder: Path) -> int:
    if folder is None or not folder.exists():
        return 0
    if RECURSIVE:
        it = folder.rglob("*")
    else:
        it = folder.iterdir()
    return sum(1 for f in it if f.is_file() and f.suffix.lower() in IMAGE_EXTS)

# --------- per-type counts ---------
rows = []  # (type_name, sickled, non_sickled)
type_dirs = [d for d in ROOT.iterdir() if is_type_dir(d)]
for tdir in sorted(type_dirs, key=lambda p: p.name):
    s_dir, n_dir = find_class_dirs(tdir)
    s_cnt = count_images(s_dir)
    n_cnt = count_images(n_dir)
    rows.append((tdir.name, s_cnt, n_cnt))

# --------- All/ counts ---------
ALL_DIR = ROOT / "All"
ALL_SICKLED = ALL_DIR / "Sickled"
ALL_NON = ALL_DIR / "Non-sickled"
all_s_cnt = count_images(ALL_SICKLED)
all_n_cnt = count_images(ALL_NON)

# --------- print table ---------
w_type = max([len(r[0]) for r in rows] + [4])
hdr = f"{'Type':<{w_type}}  {'Sickled':>8}  {'Non-sickled':>12}  {'Total':>8}"
print(hdr)
print("-" * len(hdr))
sum_s = sum(r[1] for r in rows)
sum_n = sum(r[2] for r in rows)
for t, s, n in rows:
    print(f"{t:<{w_type}}  {s:>8}  {n:>12}  {s+n:>8}")
print("-" * len(hdr))
print(f"{'SUM(types)':<{w_type}}  {sum_s:>8}  {sum_n:>12}  {sum_s+sum_n:>8}")

print("\nAll/ folder counts:")
print(f"  All/Sickled:     {all_s_cnt}")
print(f"  All/Non-sickled: {all_n_cnt}")
print(f"  All/Total:       {all_s_cnt + all_n_cnt}")

# --------- quick validation ---------
ok_s = (all_s_cnt == sum_s)
ok_n = (all_n_cnt == sum_n)
print("\nValidation:")
print(f"  Sickled   match? {'OK' if ok_s else 'MISMATCH'}  (All={all_s_cnt}, SumTypes={sum_s})")
print(f"  Non-sickled match? {'OK' if ok_n else 'MISMATCH'}  (All={all_n_cnt}, SumTypes={sum_n})")


Type   Sickled   Non-sickled     Total
--------------------------------------
A         5862          7638     13500
B         3368          2976      6344
C         1370          1042      2412
D          590           188       778
E          122           414       536
F          346          1080      1426
G         1442           578      2020
--------------------------------------
SUM(types)     13100         13916     27016

All/ folder counts:
  All/Sickled:     13100
  All/Non-sickled: 13916
  All/Total:       27016

Validation:
  Sickled   match? OK  (All=13100, SumTypes=13100)
  Non-sickled match? OK  (All=13916, SumTypes=13916)


In [27]:
from pathlib import Path
import csv

# === EDIT THIS to your dataset root ===
ROOT = Path(r"/Users/vanhorin/Documents/Wang/Paired dataset_10122025")
PAIR_DIRS = [ROOT / "All" / "Sickled", ROOT / "All" / "Non-sickled"]

# Consider these files only
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

# Pair suffixes
SUF_0, SUF_1 = "frame0", "frame1"

# If your All/* has nested subfolders, flip this to True
RECURSIVE = False

def split_pair_key(path: Path):
    """
    If filename ends with ...frame0.* or ...frame1.*, return (base, idx)
    where base is the part before 'frame{0,1}' and idx in {0,1}.
    Otherwise return None.
    """
    stem = path.stem
    if stem.endswith(SUF_0):
        return stem[: -len(SUF_0)], 0
    if stem.endswith(SUF_1):
        return stem[: -len(SUF_1)], 1
    return None

def iter_images(folder: Path):
    if not folder.exists():
        return
    it = folder.rglob("*") if RECURSIVE else folder.iterdir()
    for p in it:
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            yield p

def analyze_dir(cls_dir: Path):
    """
    Returns:
      summary: dict with counts
      unpaired: list of dict rows for unpaired files
      non_pattern: list of files not matching frame0/frame1 pattern
    """
    bucket = {}  # base -> {0: [paths], 1: [paths]}
    non_pattern = []
    total_files = 0

    for p in iter_images(cls_dir):
        total_files += 1
        parsed = split_pair_key(p)
        if parsed is None:
            non_pattern.append(p)
            continue
        base, idx = parsed
        d = bucket.setdefault(base, {0: [], 1: []})
        d[idx].append(p)

    complete_pairs = 0
    unpaired = []

    for base, sides in bucket.items():
        has0 = len(sides[0]) > 0
        has1 = len(sides[1]) > 0
        if has0 and has1:
            # Count a pair as min(count0, count1) complete pairs; remainder are effectively unpaired
            complete_pairs += min(len(sides[0]), len(sides[1]))
            # Excess on either side are unpaired
            if len(sides[0]) > len(sides[1]):
                for p in sides[0][len(sides[1]):]:
                    unpaired.append({"class_dir": cls_dir.name, "base": base, "present": "frame0", "missing": "frame1", "path": str(p)})
            elif len(sides[1]) > len(sides[0]):
                for p in sides[1][len(sides[0]):]:
                    unpaired.append({"class_dir": cls_dir.name, "base": base, "present": "frame1", "missing": "frame0", "path": str(p)})
        else:
            # Everything present for this base is unpaired (missing the other side)
            missing = "frame0" if has1 and not has0 else ("frame1" if has0 and not has1 else "frame0&frame1")
            present_side = "frame1" if has1 else ("frame0" if has0 else "none")
            for idx_side in (0,1):
                for p in sides[idx_side]:
                    unpaired.append({"class_dir": cls_dir.name, "base": base, "present": f"frame{idx_side}", "missing": missing, "path": str(p)})

    summary = {
        "class_dir": cls_dir.name,
        "total_files": total_files,
        "pattern_files": sum(len(v[0]) + len(v[1]) for v in bucket.values()),
        "non_pattern_files": len(non_pattern),
        "bases": len(bucket),
        "complete_pairs": complete_pairs,
        "unpaired_files": len(unpaired),
    }
    return summary, unpaired, non_pattern

all_summaries = []
all_unpaired = []
all_nonpattern = []

for d in PAIR_DIRS:
    if not d.exists():
        print(f"[WARN] Missing folder: {d}")
        continue
    summary, unpaired, non_pattern = analyze_dir(d)
    all_summaries.append(summary)
    all_unpaired.extend(unpaired)
    all_nonpattern.extend([{"class_dir": d.name, "path": str(p)} for p in non_pattern])

# ---- Print summaries ----
print("Pair check in All/*:")
for s in all_summaries:
    print(
        f"- {s['class_dir']}: total={s['total_files']}, "
        f"pattern={s['pattern_files']}, non-pattern={s['non_pattern_files']}, "
        f"bases={s['bases']}, complete_pairs={s['complete_pairs']}, "
        f"UNPAIRED_FILES={s['unpaired_files']}"
    )

# Show a few unpaired samples
MAX_SHOW = 20
if all_unpaired:
    print(f"\nUnpaired examples (showing up to {MAX_SHOW}):")
    for row in all_unpaired[:MAX_SHOW]:
        print(f"[{row['class_dir']}] base='{row['base']}' present={row['present']} missing={row['missing']} -> {row['path']}")
else:
    print("\nNo unpaired files found among pattern-matching files.")

# Show a few non-pattern files
if all_nonpattern:
    print(f"\nFiles not matching '*frame0|frame1*' pattern (up to {MAX_SHOW}):")
    for row in all_nonpattern[:MAX_SHOW]:
        print(f"[{row['class_dir']}] {row['path']}")

Pair check in All/*:
- Sickled: total=13100, pattern=13100, non-pattern=0, bases=6554, complete_pairs=6546, UNPAIRED_FILES=8
- Non-sickled: total=13916, pattern=13916, non-pattern=0, bases=6967, complete_pairs=6949, UNPAIRED_FILES=18

Unpaired examples (showing up to 20):
[Sickled] base='A__2193-osi-30per-4_id168.0_' present=frame0 missing=frame1 -> /Users/vanhorin/Documents/Wang/Paired dataset_10122025/All/Sickled/A__2193-osi-30per-4_id168.0_frame0.png
[Sickled] base='B__2193-osi-0per-2_id143.0_' present=frame0 missing=frame1 -> /Users/vanhorin/Documents/Wang/Paired dataset_10122025/All/Sickled/B__2193-osi-0per-2_id143.0_frame0.png
[Sickled] base='C__4th_id219.0_' present=frame1 missing=frame0 -> /Users/vanhorin/Documents/Wang/Paired dataset_10122025/All/Sickled/C__4th_id219.0_frame1.png
[Sickled] base='C__4th_id229.0_' present=frame0 missing=frame1 -> /Users/vanhorin/Documents/Wang/Paired dataset_10122025/All/Sickled/C__4th_id229.0_frame0.png
[Sickled] base='B__2193-osi-0per-3_id29.0

In [30]:
from pathlib import Path

# === EDIT THIS to your dataset root ===
ROOT = Path(r"/Users/vanhorin/Documents/Wang/Paired dataset_10122025")

# Consider these files only
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

# Pair suffixes
SUF_0, SUF_1 = "frame0", "frame1"

# If your type folders contain nested subfolders inside Sickled/Non-sickled, set True
RECURSIVE = False

def is_type_dir(p: Path) -> bool:
    return p.is_dir() and p.name.lower() != "all"

def class_dirs(type_dir: Path):
    subs = [sd for sd in type_dir.iterdir() if sd.is_dir()]
    sickled   = next((sd for sd in subs if sd.name.lower().startswith("sickled")), None)
    nonsickle = next((sd for sd in subs if sd.name.lower().startswith("non-sickled")), None)
    return sickled, nonsickle

def iter_images(folder: Path):
    if not folder or not folder.exists():
        return
    it = folder.rglob("*") if RECURSIVE else folder.iterdir()
    for p in it:
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            yield p

def split_pair_key(path: Path):
    """
    If filename ends with ...frame0.* or ...frame1.*, return (base, idx)
    where base is the part before 'frame{0,1}' and idx in {0,1}.
    Otherwise return None.
    """
    stem = path.stem
    if stem.endswith(SUF_0):
        return stem[: -len(SUF_0)], 0
    if stem.endswith(SUF_1):
        return stem[: -len(SUF_1)], 1
    return None

def complete_pairs_in_folder(folder: Path):
    """
    Count complete pairs within ONE class folder (Sickled OR Non-sickled):
      For each base: pairs = min(#frame0, #frame1).
    Returns (complete_pairs, bases_count, nonpattern_files).
    """
    bucket = {}  # base -> {0:count, 1:count}
    nonpattern = 0
    for p in iter_images(folder):
        parsed = split_pair_key(p)
        if parsed is None:
            nonpattern += 1
            continue
        base, idx = parsed
        d = bucket.setdefault(base, {0:0, 1:0})
        d[idx] += 1
    complete_pairs = sum(min(d[0], d[1]) for d in bucket.values())
    return complete_pairs, len(bucket), nonpattern

# ---------- per-type totals ----------
rows = []  # (TypeName, sickled_pairs, nonsickled_pairs, total_pairs, sickled_bases, nonsickled_bases, np_s, np_n)
type_dirs = [d for d in ROOT.iterdir() if is_type_dir(d)]

for tdir in sorted(type_dirs, key=lambda p: p.name):
    s_dir, n_dir = class_dirs(tdir)

    s_pairs = s_bases = s_nonp = 0
    n_pairs = n_bases = n_nonp = 0

    if s_dir and s_dir.exists():
        s_pairs, s_bases, s_nonp = complete_pairs_in_folder(s_dir)
    if n_dir and n_dir.exists():
        n_pairs, n_bases, n_nonp = complete_pairs_in_folder(n_dir)

    rows.append((tdir.name, s_pairs, n_pairs, s_pairs + n_pairs, s_bases, n_bases, s_nonp, n_nonp))

# ---------- print ----------
if not rows:
    print("No cell-type folders found.")
else:
    w = max(len(r[0]) for r in rows + [("Type",0,0,0,0,0,0,0)])
    header = f"{'Type':<{w}}  {'Pairs(S)':>10}  {'Pairs(N)':>10}  {'Pairs(Total)':>13}  {'Bases(S)':>9}  {'Bases(N)':>9}  {'NonPat(S)':>10}  {'NonPat(N)':>10}"
    print(header)
    print("-" * len(header))

    g_s_pairs = g_n_pairs = g_total_pairs = 0
    g_s_bases = g_n_bases = 0
    g_s_np = g_n_np = 0

    for name, sp, np_, tp, sb, nb, snp, nnp in rows:
        g_s_pairs += sp; g_n_pairs += np_; g_total_pairs += tp
        g_s_bases += sb; g_n_bases += nb
        g_s_np += snp; g_n_np += nnp
        print(f"{name:<{w}}  {sp:>10}  {np_:>10}  {tp:>13}  {sb:>9}  {nb:>9}  {snp:>10}  {nnp:>10}")

    print("-" * len(header))
    print(f"{'SUM':<{w}}  {g_s_pairs:>10}  {g_n_pairs:>10}  {g_total_pairs:>13}  {g_s_bases:>9}  {g_n_bases:>9}  {g_s_np:>10}  {g_n_np:>10}")

    print("\nNotes:")
    print("• Pairs(S/N) are counted WITHIN each class folder: Σ_base min(#frame0, #frame1).")
    print("• NonPat = files not matching the '*frame0|frame1*' naming convention.")


Type    Pairs(S)    Pairs(N)   Pairs(Total)   Bases(S)   Bases(N)   NonPat(S)   NonPat(N)
-----------------------------------------------------------------------------------------
A           2930        3815           6745       2932       3823           0           0
B           1682        1486           3168       1686       1490           0           0
C            684         519           1203        686        523           0           0
D            295          94            389        295         94           0           0
E             61         207            268         61        207           0           0
F            173         539            712        173        541           0           0
G            721         289           1010        721        289           0           0
-----------------------------------------------------------------------------------------
SUM         6546        6949          13495       6554       6967           0           0

Notes:
• 

In [9]:
from pathlib import Path

# === EDIT THIS to your dataset root ===
ROOT = Path(r"/Users/vanhorin/Documents/Wang/Paired dataset_10122025")

# Consider these image extensions
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

# Pair suffixes
SUF_0, SUF_1 = "frame0", "frame1"

# If Sickled/Non-sickled contain nested subfolders, set True
RECURSIVE = False

def is_type_dir(p: Path) -> bool:
    return p.is_dir() and p.name.lower() != "all"

def class_dirs(type_dir: Path):
    subs = [sd for sd in type_dir.iterdir() if sd.is_dir()]
    sickled   = next((sd for sd in subs if sd.name.lower().startswith("sickled")), None)
    nonsickle = next((sd for sd in subs if sd.name.lower().startswith("non-sickled")), None)
    return sickled, nonsickle

def iter_images(folder: Path):
    if not folder or not folder.exists():
        return
    it = folder.rglob("*") if RECURSIVE else folder.iterdir()
    for p in it:
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            yield p

def split_pair_key(path: Path):
    """Return (base, idx) if stem ends with frame0/frame1, else None."""
    s = path.stem
    if s.endswith(SUF_0): return s[:-len(SUF_0)], 0
    if s.endswith(SUF_1): return s[:-len(SUF_1)], 1
    return None

def count_unmatched_in_folder(folder: Path):
    """
    Returns:
      unmatched_bases: count of bases where not (has frame0 and frame1)
      total_bases: number of distinct bases (pattern-matching files)
      nonpattern: files not matching frame0/frame1
    Notes:
      If there are duplicates (e.g., 2 frame0 and 1 frame1 for same base),
      we still treat that base as 'matched' (has both sides).
    """
    bucket = {}  # base -> {0:cnt, 1:cnt}
    nonpattern = 0
    for p in iter_images(folder):
        parsed = split_pair_key(p)
        if parsed is None:
            nonpattern += 1
            continue
        base, idx = parsed
        d = bucket.setdefault(base, {0:0, 1:0})
        d[idx] += 1

    unmatched_bases = sum(1 for base, d in bucket.items() if d[0] == 0 or d[1] == 0)
    return unmatched_bases, len(bucket), nonpattern

# ---------- per-type summary ----------
rows = []  # (type_name, unmatched_pairs)
type_dirs = [d for d in ROOT.iterdir() if is_type_dir(d)]

for tdir in sorted(type_dirs, key=lambda p: p.name):
    s_dir, n_dir = class_dirs(tdir)

    s_unmatched, s_bases, s_nonp = count_unmatched_in_folder(s_dir) if s_dir else (0, 0, 0)
    n_unmatched, n_bases, n_nonp = count_unmatched_in_folder(n_dir) if n_dir else (0, 0, 0)

    unmatched_total = s_unmatched + n_unmatched
    rows.append((tdir.name, unmatched_total, s_unmatched, n_unmatched, s_bases + n_bases, s_nonp + n_nonp))

# ---------- print ----------
if not rows:
    print("No type folders found.")
else:
    w = max(len(r[0]) for r in rows + [("Type",0,0,0,0,0)])  # width for type column
    header = f"{'Type':<{w}}  {'Unmatched(total)':>16}  {'Sickled':>8}  {'Non-sickled':>12}  {'Bases':>8}  {'Non-pattern':>12}"
    print(header)
    print("-" * len(header))

    grand_unmatched = 0
    grand_bases = 0
    grand_nonpattern = 0

    for name, unmatched, s_u, n_u, bases, nonp in rows:
        grand_unmatched += unmatched
        grand_bases += bases
        grand_nonpattern += nonp
        print(f"{name:<{w}}  {unmatched:>16}  {s_u:>8}  {n_u:>12}  {bases:>8}  {nonp:>12}")

    print("-" * len(header))
    print(f"{'SUM':<{w}}  {grand_unmatched:>16}  {'' :>8}  {'' :>12}  {grand_bases:>8}  {grand_nonpattern:>12}")

    print("\nNotes:")
    print("• 'Unmatched' counts bases missing either frame0 or frame1 (per naming convention).")
    print("• 'Bases' = distinct base names (pattern-matching files only).")
    print("• 'Non-pattern' = files that do not end with 'frame0' or 'frame1' before the extension.")


Type  Unmatched(total)   Sickled   Non-sickled     Bases   Non-pattern
----------------------------------------------------------------------
A                   10         2             8      6755             0
B                    8         4             4      3176             0
C                    6         2             4      1209             0
D                    0         0             0       389             0
E                    0         0             0       268             0
F                    2         0             2       714             0
G                    0         0             0      1010             0
----------------------------------------------------------------------
SUM                 26                             13521             0

Notes:
• 'Unmatched' counts bases missing either frame0 or frame1 (per naming convention).
• 'Bases' = distinct base names (pattern-matching files only).
• 'Non-pattern' = files that do not end with 'frame0' or 'frame1

In [21]:
from pathlib import Path

ROOT = Path(r"/Users/vanhorin/Documents/Wang/Paired dataset_10122025")
TYPE = "A"  # change to the type you’re checking

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
SUF_0, SUF_1 = "frame0", "frame1"
RECURSIVE = False

def iter_imgs(p):
    it = p.rglob("*") if RECURSIVE else p.iterdir()
    for f in it:
        if f.is_file() and f.suffix.lower() in IMAGE_EXTS:
            yield f

def split_base(p):
    s = p.stem
    if s.endswith(SUF_0): return s[:-len(SUF_0)], 0
    if s.endswith(SUF_1): return s[:-len(SUF_1)], 1
    return None

def collect(folder: Path):
    bucket = {}
    for f in iter_imgs(folder):
        parsed = split_base(f)
        if parsed is None: 
            continue
        b, idx = parsed
        d = bucket.setdefault(b, {0:0, 1:0})
        d[idx] += 1
    bases = set(bucket.keys())
    unmatched = sum(1 for b,d in bucket.items() if d[0]==0 or d[1]==0)
    matched   = len(bases) - unmatched
    return bases, matched, unmatched

s_dir = ROOT/TYPE/"Sickled"
n_dir = ROOT/TYPE/"Non-sickled"

S, matched_S, unmatched_S = collect(s_dir) if s_dir.exists() else (set(),0,0)
N, matched_N, unmatched_N = collect(n_dir) if n_dir.exists() else (set(),0,0)

sum_pairs   = matched_S + matched_N          # no duplicates assumption
bases_union = len(S | N)
overlap     = len(S & N)

print(f"{TYPE} — S_bases={len(S)} N_bases={len(N)} overlap={overlap}")
print(f"matched_S={matched_S} unmatched_S={unmatched_S}")
print(f"matched_N={matched_N} unmatched_N={unmatched_N}")
print(f"sum_pairs={sum_pairs}, bases_union={bases_union}")
print(f"sum_pairs - bases_union = {sum_pairs - bases_union}  "
      f"(should equal overlap - (unmatched_S+unmatched_N) = "
      f"{overlap - (unmatched_S + unmatched_N)})")


A — S_bases=2932 N_bases=3823 overlap=15
matched_S=2930 unmatched_S=2
matched_N=3815 unmatched_N=8
sum_pairs=6745, bases_union=6740
sum_pairs - bases_union = 5  (should equal overlap - (unmatched_S+unmatched_N) = 5)


In [29]:
from pathlib import Path
import re

# === EDIT THIS to your dataset root ===
ROOT = Path(r"/Users/vanhorin/Documents/Wang/Paired dataset_10122025")
TARGETS = [ROOT / "All" / "Sickled", ROOT / "All" / "Non-sickled"]

# Consider these file types (case-insensitive)
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

# Match filenames where a numeric suffix was appended AFTER frameX:
#   <anything>frame0__<num>.<ext>  or  <anything>frame1__<num>.<ext>
pat_numeric_after_frame = re.compile(r".*frame[01]__\d+$", re.IGNORECASE)

def iter_images(folder: Path):
    if not folder.exists():
        return
    for p in folder.iterdir():  # set to folder.rglob("*") if you have nested subfolders
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            yield p

total = 0
for folder in TARGETS:
    bad = []
    for p in iter_images(folder):
        stem = p.stem  # filename without extension
        if pat_numeric_after_frame.match(stem):
            bad.append(p)

    print(f"\n[{folder.name}] files with appended numeric suffix after frameX: {len(bad)}")
    for p in sorted(bad, key=lambda x: x.name):
        print("  ", p)

    total += len(bad)

print(f"\nTOTAL across All/Sickled + All/Non-sickled: {total}")



[Sickled] files with appended numeric suffix after frameX: 0

[Non-sickled] files with appended numeric suffix after frameX: 0

TOTAL across All/Sickled + All/Non-sickled: 0
